In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [35]:
import joblib

model_path = "/content/drive/MyDrive/lda_erd_model.pkl"

lda_loaded = joblib.load(model_path)

print("Model loaded successfully.")

Model loaded successfully.


In [36]:
import os

base_path = "/content/drive/MyDrive/files"

folders = [f for f in os.listdir(base_path) if f.startswith("S")]

filenumber = ['03']
data = []

for folder in sorted(folders):
    folder_path = os.path.join(base_path, folder)

    if not os.path.isdir(folder_path):
        continue

    files = os.listdir(folder_path)

    for f in files:
        if f.lower().endswith(".edf"):

            run_part = f.split("R")[-1].split(".")[0]

            if run_part in filenumber:
                full_path = os.path.join(folder_path, f)
                data.append(full_path)

# Print final single list
print(data)
print("\nTotal files:", len(data))


['/content/drive/MyDrive/files/S071/S071R03.edf', '/content/drive/MyDrive/files/S072/S072R03.edf', '/content/drive/MyDrive/files/S074/S074R03.edf', '/content/drive/MyDrive/files/S076/S076R03.edf', '/content/drive/MyDrive/files/S077/S077R03.edf', '/content/drive/MyDrive/files/S078/S078R03.edf', '/content/drive/MyDrive/files/S079/S079R03.edf', '/content/drive/MyDrive/files/S080/S080R03.edf', '/content/drive/MyDrive/files/S081/S081R03.edf', '/content/drive/MyDrive/files/S082/S082R03.edf', '/content/drive/MyDrive/files/S083/S083R03.edf', '/content/drive/MyDrive/files/S084/S084R03.edf', '/content/drive/MyDrive/files/S085/S085R03.edf', '/content/drive/MyDrive/files/S086/S086R03.edf', '/content/drive/MyDrive/files/S087/S087R03.edf', '/content/drive/MyDrive/files/S088/S088R03.edf', '/content/drive/MyDrive/files/S089/S089R03.edf', '/content/drive/MyDrive/files/S090/S090R03.edf', '/content/drive/MyDrive/files/S091/S091R03.edf', '/content/drive/MyDrive/files/S092/S092R03.edf', '/content/drive/MyD

In [37]:
reqdata=data[0]

In [38]:
reqdata

'/content/drive/MyDrive/files/S071/S071R03.edf'

In [39]:

!pip install mne

In [40]:

import mne

file_path = reqdata

raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)

print(raw)

<RawEDF | S071R03.edf, 64 x 20000 (125.0 s), ~9.8 MiB, data loaded>


In [41]:

print("Sampling Frequency:", raw.info['sfreq'])
print("Number of Channels:", len(raw.ch_names))
print("Channel Names:", raw.ch_names)

Sampling Frequency: 160.0
Number of Channels: 64
Channel Names: ['Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.', 'Fc4.', 'Fc6.', 'C5..', 'C3..', 'C1..', 'Cz..', 'C2..', 'C4..', 'C6..', 'Cp5.', 'Cp3.', 'Cp1.', 'Cpz.', 'Cp2.', 'Cp4.', 'Cp6.', 'Fp1.', 'Fpz.', 'Fp2.', 'Af7.', 'Af3.', 'Afz.', 'Af4.', 'Af8.', 'F7..', 'F5..', 'F3..', 'F1..', 'Fz..', 'F2..', 'F4..', 'F6..', 'F8..', 'Ft7.', 'Ft8.', 'T7..', 'T8..', 'T9..', 'T10.', 'Tp7.', 'Tp8.', 'P7..', 'P5..', 'P3..', 'P1..', 'Pz..', 'P2..', 'P4..', 'P6..', 'P8..', 'Po7.', 'Po3.', 'Poz.', 'Po4.', 'Po8.', 'O1..', 'Oz..', 'O2..', 'Iz..']


In [42]:

import numpy as np
import mne

# 1️⃣ Preprocessing
raw.set_eeg_reference('average', verbose=False)
raw.filter(8., 30., verbose=False)

# 2️⃣ Get events
events, event_dict = mne.events_from_annotations(raw, verbose=False)

# 3️⃣ Epoch extraction
epochs = mne.Epochs(
    raw,
    events,
    event_id={'T0':1, 'T1':2, 'T2':3},
    tmin=0.5,
    tmax=2.5,
    baseline=None,
    preload=True,
    verbose=False
)

epochs.pick_types(eeg=True)

T0 = epochs['T0'].get_data()
T1 = epochs['T1'].get_data()
T2 = epochs['T2'].get_data()

print("T0 shape:", T0.shape)
print("T1 shape:", T1.shape)
print("T2 shape:", T2.shape)

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
T0 shape: (15, 64, 321)
T1 shape: (7, 64, 321)
T2 shape: (8, 64, 321)


In [43]:
baseline_var = np.mean(np.var(T0, axis=2), axis=0)
# shape: (channels,)

In [44]:
var_T1 = np.var(T1, axis=2)   # (trials, channels)

ERD_T1 = (var_T1 - baseline_var) / baseline_var

In [45]:
lda_loaded.predict(ERD_T1)

array([2, 1, 1, 1, 1, 1, 1])

In [46]:

var_T2 = np.var(T2, axis=2)

ERD_T2 = (var_T2 - baseline_var) / baseline_var

In [47]:
lda_loaded.predict(ERD_T2)

array([2, 2, 2, 2, 1, 1, 1, 1])

In [48]:
# Compute variance for T0
var_T0 = np.var(T0, axis=2)   # (trials, channels)

# ERD for T0
ERD_T0 = (var_T0 - baseline_var) / baseline_var

In [49]:
lda_loaded.predict(ERD_T0)

array([0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0])

In [50]:
raw.set_eeg_reference('average', verbose=False)
raw.filter(8., 30., verbose=False)

data = raw.get_data()  # shape: (channels, samples)
sfreq = int(raw.info['sfreq'])

In [51]:
window_size = 4 * sfreq   # 4 seconds
n_samples = data.shape[1]

In [52]:
baseline_segment = data[:, :window_size]

baseline_var = np.var(baseline_segment, axis=1)   # (channels,)

In [56]:
predictions = []
time_points = []

for start in range(window_size, n_samples - window_size, window_size):

    segment = data[:, start:start + window_size]

    # Variance feature
    var_segment = np.var(segment, axis=1)

    # ERD computation
    erd_segment = (var_segment - baseline_var) / baseline_var

    # reshape to (1, channels)
    erd_segment = erd_segment.reshape(1, -1)

    # Predict
    pred = lda_loaded.predict(erd_segment)

    predictions.append(pred[0])
    time_points.append(start / sfreq)

print("Predictions:", predictions)

Predictions: [np.int64(1), np.int64(0), np.int64(2), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(2), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0)]


In [57]:
events, event_dict = mne.events_from_annotations(raw)

# Reverse dictionary: id -> label
id_to_label = {v: k for k, v in event_dict.items()}

# Extract only event names
event_list = [id_to_label[ev[2]] for ev in events]

print(event_list)

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
[np.str_('T0'), np.str_('T1'), np.str_('T0'), np.str_('T2'), np.str_('T0'), np.str_('T2'), np.str_('T0'), np.str_('T1'), np.str_('T0'), np.str_('T2'), np.str_('T0'), np.str_('T1'), np.str_('T0'), np.str_('T2'), np.str_('T0'), np.str_('T1'), np.str_('T0'), np.str_('T1'), np.str_('T0'), np.str_('T2'), np.str_('T0'), np.str_('T2'), np.str_('T0'), np.str_('T1'), np.str_('T0'), np.str_('T2'), np.str_('T0'), np.str_('T1'), np.str_('T0'), np.str_('T2')]
